# Syntopy Between Scales - Master Thesis

The Aim of this project is to extract mean NDVI values for BBS routes using their starting point geometry and data from Landsat extracted with GEE.

Steps:
1. Extract satellite data for routes in each year separately.
2. Check the missing values in the data in each year.
3. Merge data into one comprehensive dataset.

## Import Data

In [1]:
import ee
import geemap
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import time
from libpysal import graph
from sklearn import neighbors

/home/matejtvar/Documents/data_science/SDS_python/.pixi/envs/default/lib/python3.12/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.82.2. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, Google Cloud Python packages will raise their minimum requirements (including google-api-core, grpcio, and grpcio-status) to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


In [6]:
dat = gpd.read_file('route_dat.gpkg', engine = 'fiona')
dat['lon'] = dat.geometry.x
dat['lat'] = dat.geometry.y
dat.head()

,RTENO,Year,geometry,lon,lat
0,12404002,2017,POINT (-111.4102 49.87324),-111.41020,49.87324
1,12404002,2018,POINT (-111.4102 49.87324),-111.41020,49.87324
2,12404002,2019,POINT (-111.4102 49.87324),-111.41020,49.87324
3,12404003,2017,POINT (-112.39699 49.08995),-112.39699,49.08995
4,12404004,2017,POINT (-113.56982 49.81509),-113.56982,49.81509


In [79]:
# Check unique active routes per year
active_counts = dat.groupby("Year")["RTENO"].nunique()
print(active_counts)

Year
2017    3021
2018    3002
2019    2891
Name: RTENO, dtype: int64


In [3]:
#  Initialise Earth Engine Project ID
try:
    ee.Initialize(project='syntopypredictors2026')

except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='syntopypredictors2026')

In [4]:
connectivity_test = ee.Number(1).add(2).getInfo()
print('connectivity test: ', connectivity_test)

connectivity test:  3


In [5]:
# Import Landsat Collection 2 Tier 1 Level 2 32-Day NDVI Composite
ndvi = ee.ImageCollection('LANDSAT/COMPOSITES/C02/T1_L2_32DAY_NDVI')

## Extract NDVI for each year separately

In [134]:
years = ['2017', '2018', '2019']
final_yearly_datasets = {}  # Dictionary to hold each year

for yr in years:
  print(f'\n--- Processing Breeding Season NDVI for {yr} ---')

  # Filter routes for current year
  year_dat = dat[dat['Year'] == int(yr)].copy()
  print(f'Active routes for {yr}: {len(year_dat)}')

  # Define breeding window (May 1 - July 1)
  i_date = f'{yr}-05-01'
  f_date = f'{yr}-07-01'

  ndvi_col = (
      ee.ImageCollection('LANDSAT/COMPOSITES/C02/T1_L2_32DAY_NDVI')
      .select('NDVI')
      .filterDate(i_date, f_date)
      .mean()
  )

  # Batched extraction loop
  chunk_size = 500
  all_results = []

  for i in range(0, len(year_dat), chunk_size):
    chunk = year_dat.iloc[i : i + chunk_size]
    # Extract features for the 3 km buffer around each point
    features = []
    for _, row in chunk.iterrows():
      point = ee.Geometry.Point([row['lon'], row['lat']])
      buff = point.buffer(1000)  # 1000 m buffer
      feat = ee.Feature(buff, {'RTENO': row['RTENO']})
      features.append(feat)

    points_fc = ee.FeatureCollection(features)
    sampled_chunk = ndvi_col.reduceRegions(
        collection=points_fc, reducer=ee.Reducer.mean(), scale=30
    )

    try:
      info = sampled_chunk.getInfo()
      for f in info['features']:
        all_results.append(f['properties'])
      print(
          f'  Successfully processed rows {i} to'
          f' {min(i + chunk_size, len(year_dat))}'
      )
    except Exception as e:
      print(f'  Error on chunk starting at {i}: {e}')

    time.sleep(1)

  # Convert to DataFrame and merge back to year_dat
  df_ndvi = pd.DataFrame(all_results)
  year_merged = year_dat.merge(
      df_ndvi[['RTENO', 'mean']], on='RTENO', how='left'
  )
  year_merged = year_merged.rename(columns={"mean": "NDVI"})

  # Save the finished year into our dictionary
  final_yearly_datasets[yr] = year_merged

print('\nAll years successfully processed!')

df_all = pd.concat(final_yearly_datasets.values(), ignore_index=True)


--- Processing Breeding Season NDVI for 2017 ---
Active routes for 2017: 3021
  Successfully processed rows 0 to 500
  Successfully processed rows 500 to 1000
  Successfully processed rows 1000 to 1500
  Successfully processed rows 1500 to 2000
  Successfully processed rows 2000 to 2500
  Successfully processed rows 2500 to 3000
  Successfully processed rows 3000 to 3021

--- Processing Breeding Season NDVI for 2018 ---
Active routes for 2018: 3002
  Successfully processed rows 0 to 500
  Successfully processed rows 500 to 1000
  Successfully processed rows 1000 to 1500
  Successfully processed rows 1500 to 2000
  Successfully processed rows 2000 to 2500
  Successfully processed rows 2500 to 3000
  Successfully processed rows 3000 to 3002

--- Processing Breeding Season NDVI for 2019 ---
Active routes for 2019: 2891
  Successfully processed rows 0 to 500
  Successfully processed rows 500 to 1000
  Successfully processed rows 1000 to 1500
  Successfully processed rows 1500 to 2000
  Su

In [142]:
df_all.head()

,RTENO,Year,geometry,lon,lat,NDVI
0,12404002,2017,POINT (-111.4102 49.87324),-111.41020,49.87324,0.517917
1,12404003,2017,POINT (-112.39699 49.08995),-112.39699,49.08995,0.509571
2,12404004,2017,POINT (-113.56982 49.81509),-113.56982,49.81509,0.642169
3,12404006,2017,POINT (-110.25869 50.74668),-110.25869,50.74668,0.523844
4,12404009,2017,POINT (-113.02852 50.60147),-113.02852,50.60147,0.619111


In [141]:
dat_unique = df_all.copy()
m = dat_unique.explore('NDVI', tiles='OpenStreetMap HOT', marker_kwds={"radius": 4, "fill": True})

In [131]:
# Count of missing (NaN) NDVI values per year
na_by_year = df_all.groupby("Year")["NDVI"].apply(lambda s: s.isna().sum())
print(na_by_year)

Year
2017    56
2018    34
2019    69
Name: NDVI, dtype: int64


## Eliminate NA values by combining the data

In [147]:
# Ensure Year is integer and RTENO is consistent string
wide_ndvi = (
    df_all.assign(
        Year=df_all["Year"].astype(int), RTENO=df_all["RTENO"].astype(str)
    )
    .drop_duplicates(subset=["RTENO", "Year"])
    .pivot(index="RTENO", columns="Year", values="NDVI")
    .rename(columns={2017: "ndvi_2017", 2018: "ndvi_2018", 2019: "ndvi_2019"})
)

In [150]:
wide_ndvi.head()

Year,ndvi_2017,ndvi_2018,ndvi_2019
RTENO,,,
12404002,0.517917,0.464398,0.257739
12404003,0.509571,NaN,NaN
12404004,0.642169,0.569214,NaN
12404006,0.523844,0.389811,0.391066
12404009,0.619111,0.511970,NaN


In [151]:
# 2018: anchor on 2018, else mean of available neighbors (2017 / 2019)
neighbor_mean_18 = wide_ndvi[["ndvi_2017", "ndvi_2019"]].mean(axis=1)
wide_ndvi["ndvi_2017_filled"] = (
    wide_ndvi["ndvi_2017"]
    .fillna(wide_ndvi["ndvi_2018"])
    .fillna(wide_ndvi["ndvi_2019"])
)
wide_ndvi["ndvi_2018_filled"] = wide_ndvi["ndvi_2018"].fillna(neighbor_mean_18)
wide_ndvi["ndvi_2019_filled"] = (
    wide_ndvi["ndvi_2019"]
    .fillna(wide_ndvi["ndvi_2018"])
    .fillna(wide_ndvi["ndvi_2017"])
)

In [152]:
for yr in [2017, 2018, 2019]:
  raw_col = f"ndvi_{yr}"
  filled_col = f"ndvi_{yr}_filled"
  before_na = wide_ndvi[raw_col].isna().sum()
  after_na = wide_ndvi[filled_col].isna().sum()
  rescued = before_na - after_na
  print(
      f"{yr}: raw NaNs = {before_na} | gap-filled = {rescued} | remaining NaNs"
      f" = {after_na}"
  )

2017: raw NaNs = 649 | gap-filled = 638 | remaining NaNs = 11
2018: raw NaNs = 646 | gap-filled = 635 | remaining NaNs = 11
2019: raw NaNs = 792 | gap-filled = 781 | remaining NaNs = 11


In [153]:
persistent_na = wide_ndvi[wide_ndvi["ndvi_2018_filled"].isna()]
print(persistent_na[["ndvi_2017", "ndvi_2018", "ndvi_2019"]])

Year      ndvi_2017  ndvi_2018  ndvi_2019
RTENO                                    
12457027        NaN        NaN        NaN
12457035        NaN        NaN        NaN
12468162        NaN        NaN        NaN
12476347        NaN        NaN        NaN
84002023        NaN        NaN        NaN
84002026        NaN        NaN        NaN
84025052        NaN        NaN        NaN
84044010        NaN        NaN        NaN
84044031        NaN        NaN        NaN
84044307        NaN        NaN        NaN
84090035        NaN        NaN        NaN


In [154]:
# Reshape filled columns back to long (RTENO, Year) -> NDVI_filled
filled_long = (
    wide_ndvi[["ndvi_2017_filled", "ndvi_2018_filled", "ndvi_2019_filled"]]
    .rename(
        columns={
            "ndvi_2017_filled": 2017,
            "ndvi_2018_filled": 2018,
            "ndvi_2019_filled": 2019,
        }
    )
    .stack(future_stack=True)
    .rename("NDVI_filled")
    .reset_index()
    .rename(columns={"level_1": "Year"})
)

df_all_filled = df_all.copy()
df_all_filled["RTENO"] = df_all_filled["RTENO"].astype(str)
df_all_filled["Year"] = df_all_filled["Year"].astype(int)

df_all_filled = df_all_filled.merge(filled_long, on=["RTENO", "Year"], how="left")

# Verify remaining missing values per year in the final dataset
print(
    df_all_filled.groupby("Year")["NDVI_filled"].apply(lambda s: s.isna().sum())
)

Year
2017    4
2018    1
2019    6
Name: NDVI_filled, dtype: int64


In [162]:
df_all_filled.head()
df_all_filled.loc[df_all_filled['RTENO'] == '12457026'].head()

,RTENO,Year,geometry,lon,lat,NDVI,NDVI_filled
1560,12457026,2017,POINT (-56.66654 51.72074),-56.666545,51.720739,NaN,0.108575
4554,12457026,2018,POINT (-56.66654 51.72074),-56.666545,51.720739,0.108575,0.108575
7500,12457026,2019,POINT (-56.66654 51.72074),-56.666545,51.720739,NaN,0.108575


In [ ]:
dat_unique = df_all_filled.copy()

# Inspect either raw 'NDVI' or gap-filled 'NDVI_filled'
col = "NDVI_filled"

m = dat_unique.explore(
    col,
    tiles="OpenStreetMap HOT",
    marker_kwds={"radius": 4, "fill": True},
    missing_kwds={
        "color": "red",
        "fillColor": "red",
        "fillOpacity": 0.9,
    },
)
m

In [ ]:
dat_unique = df_all_filled.copy()
m = dat_unique.explore('NDVI', tiles='OpenStreetMap HOT', marker_kwds={"radius": 4, "fill": True})
m

In [161]:
col = "NDVI_filled"
na_rows = dat_unique[dat_unique[col].isna()]

# 1. Base map with valid NDVI values
m = dat_unique.dropna(subset=[col]).explore(
    col,
    tiles="OpenStreetMap HOT",
    marker_kwds={"radius": 4, "fill": True},
    missing_kwds={"color": "red", "fillColor": "red", "fillOpacity": 0.9},
    name="Valid NDVI",
)

# 2. Overlay missing (NaN) routes so they pop visually
if not na_rows.empty:
  na_rows.explore(
      m=m,
      color="crimson",
      marker_kwds={"radius": 7, "fill": True, "fillOpacity": 0.95},
      tooltip=["RTENO", "Year", col],
      name="Missing NDVI (NaN)",
  )

m

In [167]:
df_all_filled.to_file("ndvi_data.gpkg", layer="bbs_routes", driver="GPKG") # Save to a GeoPackage
data = gpd.read_file("ndvi_data.gpkg")
data.head()

,RTENO,Year,lon,lat,NDVI,NDVI_filled,geometry
0,12404002,2017,-111.41020,49.87324,0.517917,0.517917,POINT (-111.4102 49.87324)
1,12404003,2017,-112.39699,49.08995,0.509571,0.509571,POINT (-112.39699 49.08995)
2,12404004,2017,-113.56982,49.81509,0.642169,0.642169,POINT (-113.56982 49.81509)
3,12404006,2017,-110.25869,50.74668,0.523844,0.523844,POINT (-110.25869 50.74668)
4,12404009,2017,-113.02852,50.60147,0.619111,0.619111,POINT (-113.02852 50.60147)
